# Retail Customer Churn Prediction & CLV-Based Segmentation
### End-to-End Machine Learning Pipeline 

**Business Problem:**  
In retail, acquiring a new customer costs **5–7× more** than retaining one.  
This notebook builds a complete ML pipeline that:
1. Predicts which customers will churn using behavioral features + XGBoost
2. Quantifies revenue at risk using Customer Lifetime Value (CLV)
3. Segments customers into 4 actionable priority groups

**Dataset:** UCI Online Retail — 541,909 transactions, UK-based retailer, Dec 2010 – Dec 2011  
**Currency:** All monetary values in ₹ (£1 = ₹107, fixed 2010–2011 rate)

---
## Table of Contents
1. Environment Setup & Imports
2. Data Loading & Initial Exploration
3. Data Cleaning
4. Temporal Window Definition & INR Conversion
5. RFM & Behavioral Feature Engineering
6. Churn Label Creation
7. RFM Scoring & Customer Segmentation
8. Exploratory Data Analysis (EDA)
9. Feature Selection via Mutual Information
10. Churn Separation Analysis (KDE Plots)
11. Stratified Temporal Train / Test Split
12. Predictive Modelling (LR + RF + XGBoost)
13. Model Evaluation & Overfitting Diagnosis
14. Classification Threshold Analysis
15. Model Explainability — SHAP Values
16. Production Model — Score All Customers
17. Customer Lifetime Value (CLV) & Priority Segmentation
18. Business Intelligence Dashboard
19. Model & Business Validation
20. Save All Outputs


## 1. Environment Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import math
import json
from pathlib import Path

warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)
from sklearn.feature_selection import mutual_info_classif
from xgboost import XGBClassifier

try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP available")
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed — SHAP cells will be skipped")

# ── Constants ─────────────────────────────────────────────────────────────────
GBP_TO_INR   = 107          # Fixed GBP -> INR rate (2010-2011 period)
CUTOFF_DATE  = '2011-09-30' # Observation window ends here
MID_DATE     = '2011-06-30' # For stratified temporal split
RANDOM_STATE = 42
MI_THRESHOLD = 0.07         # Mutual information cutoff for feature selection
RISK_THRESHOLD = 0.50       # Churn probability threshold
CLV_PERCENTILE = 0.65       # Top 35% = High Value customers
ONE_TIMER_PENALTY = 0.05    # 95% CLV discount for single-purchase customers

# ── Plot Style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
PALETTE      = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
SEG_COLORS   = {'Champions':'#2ecc71','Loyal':'#3498db','At Risk':'#e74c3c',
                'Dormant':'#95a5a6','Others':'#f39c12'}
PRIO_COLORS  = {'High Priority':'#e74c3c','Loyalty':'#2ecc71',
                'Nurture':'#f39c12','Low Priority':'#95a5a6'}

print(f"All libraries loaded.")
print(f"Currency: £1 = INR {GBP_TO_INR} (fixed 2010-2011 rate)")


SHAP available
All libraries loaded.
Currency: £1 = INR 107 (fixed 2010-2011 rate)


## 2. Data Loading & Initial Exploration

**What we do:** Load the raw CSV and understand its structure.  
**Why:** Before any cleaning, we need to know what we're working with — 
shape, date range, missing values, and unique counts.


In [ ]:
# ── Load raw data ────────────────────────────────────────────────────────────
# Update this path to your local file location
DATA_PATH = 'data/OnlineRetail.csv'

df = pd.read_csv(DATA_PATH, encoding='latin1')
print(f"Raw shape         : {df.shape}")
print(f"Date range        : {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")
print(f"Unique customers  : {df['CustomerID'].nunique():,}")
print(f"Unique products   : {df['StockCode'].nunique():,}")
print(f"Countries         : {df['Country'].nunique()}")
df.head()


In [ ]:
# ── Missing values summary ───────────────────────────────────────────────────
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0]
print("=== Missing Values ===")
display(missing_df)

# Key observations:
# - CustomerID: 24.93% missing -> cannot build customer profiles without ID
# - Description: 0.27% missing -> minor, won't affect customer-level aggregation


In [ ]:
# ── Data types & sample ──────────────────────────────────────────────────────
print("Data Types:")
print(df.dtypes)
print()
print("Sample rows:")
display(df.sample(5, random_state=42))


In [ ]:
# ── Country distribution ─────────────────────────────────────────────────────
country_counts = df['Country'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(country_counts.index, country_counts.values, color='#3498db', edgecolor='white')
ax.set_title('Top 10 Countries by Transaction Count')
ax.set_xlabel('Country')
ax.set_ylabel('Transactions')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(f"UK accounts for {country_counts['United Kingdom']/len(df)*100:.1f}% of all transactions")


## 3. Data Cleaning

**What we do:** Apply 5 cleaning steps to remove unusable rows.  
**Why each step matters:**

| Step | Reason |
|------|--------|
| Drop missing CustomerID | Cannot build customer profiles without an ID |
| Remove cancelled invoices (prefix 'C') | Cancellations distort spend & frequency |
| Remove negative Quantity / UnitPrice | Data entry errors / already-handled returns |
| 99th-percentile outlier cap | Prevents bulk-order wholesalers skewing the model |
| Drop exact duplicates | System logging artefacts |


In [ ]:
# ── Type casting ─────────────────────────────────────────────────────────────
df_clean = df.copy()
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')
df_clean['CustomerID']  = df_clean['CustomerID'].astype('Int64')
df_clean['InvoiceNo']   = df_clean['InvoiceNo'].astype(str)
df_clean['StockCode']   = df_clean['StockCode'].astype(str)
print("Type casting done.")


In [ ]:
# ── Step 1: Drop missing CustomerID ─────────────────────────────────────────
before = len(df_clean)
df_clean = df_clean.dropna(subset=['CustomerID'])
print(f"Step 1 — Drop missing CustomerID   : {before - len(df_clean):,} rows removed")

# ── Step 2: Remove cancellations ─────────────────────────────────────────────
before = len(df_clean)
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]
print(f"Step 2 — Remove cancellations      : {before - len(df_clean):,} rows removed")

# ── Step 3: Remove invalid qty/price ─────────────────────────────────────────
before = len(df_clean)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f"Step 3 — Remove invalid qty/price  : {before - len(df_clean):,} rows removed")

# ── Step 4: 99th-percentile outlier cap ──────────────────────────────────────
qty_cap   = df_clean['Quantity'].quantile(0.99)
price_cap = df_clean['UnitPrice'].quantile(0.99)
before = len(df_clean)
df_clean = df_clean[(df_clean['Quantity'] <= qty_cap) & (df_clean['UnitPrice'] <= price_cap)]
print(f"Step 4 — 99th-pct cap              : {before - len(df_clean):,} rows removed")
print(f"         (Quantity <= {qty_cap:.0f}, UnitPrice <= {price_cap:.2f})")

# ── Step 5: Drop exact duplicates ────────────────────────────────────────────
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Step 5 — Drop exact duplicates     : {before - len(df_clean):,} rows removed")

print(f"\nClean shape  : {df_clean.shape}")
print(f"Customers    : {df_clean['CustomerID'].nunique():,}")
print(f"Date range   : {df_clean['InvoiceDate'].min().date()} -> {df_clean['InvoiceDate'].max().date()}")


In [ ]:
# ── Visualise cleaning impact ────────────────────────────────────────────────
labels = ['Raw Data', 'After Cleaning']
sizes  = [len(df), len(df_clean)]
colors = ['#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, sizes, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Row Count Before vs After Cleaning')
axes[0].set_ylabel('Number of Rows')
for i, v in enumerate(sizes):
    axes[0].text(i, v + 3000, f'{v:,}', ha='center', fontweight='bold')

removed_by_step = {
    'Missing\nCustomerID': 135080,
    'Cancellations': 8905,
    'Invalid\nQty/Price': 40,
    'Outlier\nCap': 7624,
    'Duplicates': 5179,
}
axes[1].bar(removed_by_step.keys(), removed_by_step.values(),
            color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_title('Rows Removed per Cleaning Step')
axes[1].set_ylabel('Rows Removed')
plt.tight_layout()
plt.show()


## 4. Temporal Window Definition & INR Conversion

**Why temporal separation?**  
Random splitting leaks future behaviour into training features. Instead we use a hard cutoff:
- **Observation window (features):** Dec 2010 – Sep 2011
- **Prediction window (churn label):** Oct – Dec 2011

Customers absent from the future window are labelled `Churn = 1`.

**INR conversion is applied here** — before any feature is computed — so every downstream ₹ figure is consistent throughout the notebook.


In [ ]:
# ── Create temporal windows ──────────────────────────────────────────────────
CUTOFF = pd.to_datetime(CUTOFF_DATE)
df_obs    = df_clean[df_clean['InvoiceDate'] <= CUTOFF].copy()
df_future = df_clean[df_clean['InvoiceDate'] >  CUTOFF].copy()

# Apply GBP -> INR conversion BEFORE feature engineering
# This ensures all downstream monetary values are in INR
df_obs['UnitPrice']  = df_obs['UnitPrice'] * GBP_TO_INR
df_obs['TotalPrice'] = df_obs['Quantity']  * df_obs['UnitPrice']

print(f"Observation window : {df_obs['InvoiceDate'].min().date()} -> {df_obs['InvoiceDate'].max().date()}")
print(f"  Customers        : {df_obs['CustomerID'].nunique():,}")
print(f"  Transactions     : {len(df_obs):,}")
print(f"  Total Revenue    : INR {df_obs['TotalPrice'].sum():,.0f}")
print()
print(f"Future window      : {df_future['InvoiceDate'].min().date()} -> {df_future['InvoiceDate'].max().date()}")
print(f"  Customers        : {df_future['CustomerID'].nunique():,}")
print(f"  (Used ONLY to create churn labels — NOT for features)")


In [ ]:
# ── Monthly revenue trend ────────────────────────────────────────────────────
df_obs['YearMonth'] = df_obs['InvoiceDate'].dt.to_period('M')
monthly = df_obs.groupby('YearMonth').agg(
    Revenue   = ('TotalPrice', 'sum'),
    Customers = ('CustomerID', 'nunique'),
    Orders    = ('InvoiceNo',  'nunique'),
).reset_index()
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Observation Window Business Trends', fontsize=14, fontweight='bold')

axes[0].bar(monthly['YearMonth_str'], monthly['Revenue'] / 1e6,
            color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Monthly Revenue (INR Millions)')
axes[0].set_ylabel('Revenue (INR M)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly['YearMonth_str'], monthly['Customers'],
             'o-', color='#2ecc71', linewidth=2, markersize=6)
axes[1].set_title('Monthly Active Customers')
axes[1].set_ylabel('Unique Customers')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 5. RFM & Behavioral Feature Engineering

**What is RFM?**
- **Recency** — How many days since the customer last bought? (lower = more recent)
- **Frequency** — How many distinct orders did they place?
- **Monetary** — How much did they spend in total (INR)?

**6 additional behavioral features:**

| Feature | Description | Business Signal |
|---------|-------------|-----------------|
| ActiveDays | Distinct days with a purchase | Engagement depth |
| ActiveMonths | Distinct calendar months active | **Strongest churn predictor** |
| CustomerAge | Days from first purchase to snapshot | Customer seniority |
| AvgBasketSize | Mean order value (INR) | Spending habit |
| ProductDiversity | Unique SKUs purchased | Catalogue engagement |
| AvgPurchaseInterval | Mean gap (days) between orders | Purchase rhythm |

**Log transforms** are applied to right-skewed features (Monetary, Frequency, AvgBasketSize, AvgPurchaseInterval) using `log1p(x)` to compress extreme ranges.


In [ ]:
# ── Snapshot date (reference point for Recency) ──────────────────────────────
snapshot_date = df_obs['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")
print("(All Recency values measured from this date)")


In [ ]:
# ── Core RFM ─────────────────────────────────────────────────────────────────
rfm = df_obs.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum'),
).reset_index()

print(f"RFM shape: {rfm.shape}")
print()
print("RFM Summary Statistics (Monetary in INR):")
display(rfm[['Recency','Frequency','Monetary']].describe().round(2))


In [ ]:
# ── Behavioral features ──────────────────────────────────────────────────────
df_feat = df_obs.copy()
df_feat['InvoiceDay'] = df_feat['InvoiceDate'].dt.date
df_feat['YearMonth']  = df_feat['InvoiceDate'].dt.to_period('M')

# ActiveDays: how many distinct calendar days did the customer shop?
active_days = df_feat.groupby('CustomerID')['InvoiceDay'].nunique().rename('ActiveDays')

# ActiveMonths: how many distinct calendar months?
active_months = df_feat.groupby('CustomerID')['YearMonth'].nunique().rename('ActiveMonths')

# CustomerAge: days between first purchase and snapshot
first_purch  = df_feat.groupby('CustomerID')['InvoiceDate'].min().rename('FirstPurchase_')
customer_age = ((snapshot_date - first_purch).dt.days).rename('CustomerAge')

# AvgBasketSize: mean total per invoice (INR)
inv_totals  = df_feat.groupby(['CustomerID','InvoiceNo'])['TotalPrice'].sum()
avg_basket  = inv_totals.groupby(level='CustomerID').mean().rename('AvgBasketSize')

# ProductDiversity: unique SKUs purchased
product_div = df_feat.groupby('CustomerID')['StockCode'].nunique().rename('ProductDiversity')

# AvgPurchaseInterval: mean gap (days) between consecutive orders
df_sorted = df_feat.sort_values(['CustomerID','InvoiceDate'])
df_sorted['PrevPurchase'] = df_sorted.groupby('CustomerID')['InvoiceDate'].shift(1)
df_sorted['PurchaseGap']  = (df_sorted['InvoiceDate'] - df_sorted['PrevPurchase']).dt.days
avg_gap = df_sorted.groupby('CustomerID')['PurchaseGap'].mean().rename('AvgPurchaseInterval')

# ── Join all features onto RFM ────────────────────────────────────────────────
customer_df = rfm.set_index('CustomerID')
for s in [active_days, active_months, customer_age, avg_basket, product_div, avg_gap]:
    customer_df = customer_df.join(s, how='left')
customer_df = customer_df.reset_index()

# Fix single-purchase customers (no interval) — use CustomerAge as proxy
customer_df['AvgPurchaseInterval'] = customer_df['AvgPurchaseInterval'].fillna(
    customer_df['CustomerAge']
)

print(f"Feature matrix shape: {customer_df.shape}")
print(f"Columns: {customer_df.columns.tolist()}")


In [ ]:
# ── Log transforms ───────────────────────────────────────────────────────────
# Right-skewed features are compressed using log1p to help ML models
for col in ['Monetary','Frequency','AvgBasketSize','AvgPurchaseInterval']:
    customer_df[f'{col}_log'] = np.log1p(customer_df[col])

print("Log-transformed features added:")
print([c for c in customer_df.columns if c.endswith('_log')])
print()

# Why log? Compare scales before and after:
print("Before log transform (Monetary range):")
print(f"  Min: INR {customer_df['Monetary'].min():,.0f}")
print(f"  Max: INR {customer_df['Monetary'].max():,.0f}")
print(f"  Range ratio: {customer_df['Monetary'].max()/customer_df['Monetary'].min():,.0f}x")
print()
print("After log transform (Monetary_log range):")
print(f"  Min: {customer_df['Monetary_log'].min():.2f}")
print(f"  Max: {customer_df['Monetary_log'].max():.2f}")
print(f"  Range ratio: {customer_df['Monetary_log'].max()/customer_df['Monetary_log'].min():.1f}x  <- much more manageable!")


In [ ]:
# ── Feature distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Feature Distributions (Observation Window)', fontsize=14, fontweight='bold')
axes = axes.flatten()

features_to_plot = [
    ('Recency',             'Days Since Last Purchase',  '#3498db'),
    ('Frequency',           'Number of Orders',          '#2ecc71'),
    ('Monetary',            'Total Spend (INR)',          '#e74c3c'),
    ('ActiveMonths',        'Active Months',              '#f39c12'),
    ('ProductDiversity',    'Unique SKUs',                '#9b59b6'),
    ('AvgPurchaseInterval', 'Avg Days Between Orders',   '#1abc9c'),
]
for ax, (col, xlabel, color) in zip(axes, features_to_plot):
    ax.hist(customer_df[col], bins=40, color=color, edgecolor='white', linewidth=0.5)
    ax.set_title(col)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Customers')
    med = customer_df[col].median()
    ax.axvline(med, color='black', linestyle='--', linewidth=1.2, label=f'Median: {med:.0f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 6. Churn Label Creation

**Definition:** A customer is `Churn = 1` if they made **zero purchases** in Oct–Dec 2011 (the future window).

This is a **behavioral definition** — it captures real disengagement without requiring customers to formally cancel.

**Why this window?** 3 months is long enough to confirm absence but short enough to remain predictive for retention campaigns.


In [ ]:
# ── Create churn label ───────────────────────────────────────────────────────
future_customers = df_future['CustomerID'].unique()

# Churn = 1 if NOT in the future window
customer_df['Churn'] = (~customer_df['CustomerID'].isin(future_customers)).astype(int)

# Distribution
dist = customer_df['Churn'].value_counts()
dist_pct = customer_df['Churn'].value_counts(normalize=True).mul(100).round(1)

print("=== Churn Label Distribution ===")
print(f"Retained (Churn=0) : {dist[0]:,} customers ({dist_pct[0]:.1f}%)")
print(f"Churned  (Churn=1) : {dist[1]:,} customers ({dist_pct[1]:.1f}%)")
print()
print("Classes are roughly balanced — no aggressive resampling needed!")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Churn Label Distribution', fontsize=13, fontweight='bold')

axes[0].pie([dist[0], dist[1]], labels=['Retained', 'Churned'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Overall Churn Rate')

axes[1].bar(['Retained', 'Churned'], [dist[0], dist[1]],
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_ylabel('Customers')
axes[1].set_title('Customer Count by Churn Status')
for i, v in enumerate([dist[0], dist[1]]):
    axes[1].text(i, v + 20, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


## 7. RFM Scoring & Customer Segmentation

**Scoring 1–5:** Each RFM dimension is split into quintiles.  
- **R_score:** Reversed — low Recency (recent buyer) → score 5  
- **F_score:** High Frequency → score 5  
- **M_score:** High Monetary → score 5

**Segment Rules (updated):**
```
R=5, F≥4, M≥4  →  Champions  (best customers)
F≥4, R≥3       →  Loyal      (buy frequently)
R≤2, F≥2       →  At Risk    (lapsing regular buyers)
R=1             →  Dormant    (very long absence)
else            →  Others
```


In [ ]:
# ── RFM Scoring ──────────────────────────────────────────────────────────────
rfm_scored = customer_df.copy()
rfm_scored['R_score'] = pd.qcut(rfm_scored['Recency'],                              5, labels=[5,4,3,2,1])
rfm_scored['F_score'] = pd.qcut(rfm_scored['Frequency'].rank(method='first'),       5, labels=[1,2,3,4,5])
rfm_scored['M_score'] = pd.qcut(rfm_scored['Monetary'],                             5, labels=[1,2,3,4,5])
rfm_scored['RFM_Score'] = (rfm_scored['R_score'].astype(str) +
                            rfm_scored['F_score'].astype(str) +
                            rfm_scored['M_score'].astype(str))

print("RFM Score distribution (first 10 unique scores):")
print(rfm_scored['RFM_Score'].value_counts().head(10))


In [ ]:
# ── Segment assignment ───────────────────────────────────────────────────────
def assign_segment(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    if r == 5 and f >= 4 and m >= 4:
        return 'Champions'
    elif f >= 4 and r >= 3:
        return 'Loyal'
    elif r <= 2 and f >= 2:
        return 'At Risk'
    elif r == 1:
        return 'Dormant'
    else:
        return 'Others'

rfm_scored['Segment'] = rfm_scored.apply(assign_segment, axis=1)
customer_df = rfm_scored.copy()

# ── Segment summary ───────────────────────────────────────────────────────────
seg_counts = customer_df['Segment'].value_counts()
print("=== Segment Distribution ===")
print(seg_counts.to_string())
print()

# Churn rate per segment
print("=== Churn Rate by Segment ===")
seg_churn = customer_df.groupby('Segment')['Churn'].mean().sort_values(ascending=True)
for seg, rate in seg_churn.items():
    bar = '#' * int(rate * 30)
    print(f"  {seg:<12} {rate:.1%}  {bar}")


In [ ]:
# ── Segment visualisations ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Segmentation Overview', fontsize=14, fontweight='bold')

colors_ord = [SEG_COLORS.get(s, '#aaa') for s in seg_counts.index]
bars = axes[0].bar(seg_counts.index, seg_counts.values, color=colors_ord, edgecolor='white')
axes[0].set_title('Customer Count by Segment')
axes[0].set_ylabel('Customers')
for b, v in zip(bars, seg_counts.values):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+10,
                 f'{v:,}', ha='center', fontsize=10, fontweight='bold')

seg_rev = customer_df.groupby('Segment')['Monetary'].sum().sort_values(ascending=False) / 1000
rev_cols = [SEG_COLORS.get(s, '#aaa') for s in seg_rev.index]
bars2 = axes[1].bar(seg_rev.index, seg_rev.values, color=rev_cols, edgecolor='white')
axes[1].set_title('Revenue by Segment (INR thousands)')
axes[1].set_ylabel('Revenue (INR k)')
for b, v in zip(bars2, seg_rev.values):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+1,
                 f'INR{v:,.0f}k', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 8. Exploratory Data Analysis

**Goal:** Understand the data distributions and validate that features 
make business sense before training any model.


In [ ]:
# ── RFM distributions ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('RFM Distributions (Observation Window)', fontsize=14, fontweight='bold')

for ax, col, color, xlabel in zip(
    axes,
    ['Recency', 'Frequency', 'Monetary'],
    ['#3498db', '#2ecc71', '#e74c3c'],
    ['Days Since Last Purchase', 'Number of Orders', 'Total Spend (INR)'],
):
    ax.hist(customer_df[col], bins=40, color=color, edgecolor='white', linewidth=0.5)
    ax.set_title(col)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Customers')
    med = customer_df[col].median()
    ax.axvline(med, color='black', linestyle='--', linewidth=1.2, label=f'Median: {med:,.0f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Key stats (Monetary in INR):")
display(customer_df[['Recency','Frequency','Monetary']].describe().round(2))


In [ ]:
# ── Churn vs features (box plots) ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Feature Distribution by Churn Status', fontsize=14, fontweight='bold')
axes = axes.flatten()

features_box = ['Recency', 'Frequency', 'Monetary_log', 
                'ActiveMonths', 'ProductDiversity', 'AvgPurchaseInterval']
for ax, feat in zip(axes, features_box):
    data_ret = customer_df[customer_df['Churn'] == 0][feat]
    data_chu = customer_df[customer_df['Churn'] == 1][feat]
    ax.boxplot([data_ret, data_chu], labels=['Retained', 'Churned'],
               patch_artist=True,
               boxprops=dict(facecolor='#dbeafe', color='#2563eb'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    ax.set_title(feat)
    ax.set_ylabel('Value')

plt.tight_layout()
plt.show()


In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────────
feat_cols = ['Recency','Frequency','Monetary','ActiveDays','ActiveMonths',
             'AvgBasketSize','ProductDiversity','AvgPurchaseInterval','Churn']
corr = customer_df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key correlations with Churn:")
print(corr['Churn'].sort_values(ascending=False).round(3).to_string())


In [ ]:
# ── Cohort retention analysis ─────────────────────────────────────────────────
df_cohort = df_obs.copy()
df_cohort['InvoiceMonth'] = df_cohort['InvoiceDate'].dt.to_period('M')
first_month = df_cohort.groupby('CustomerID')['InvoiceMonth'].min().rename('CohortMonth')
df_cohort = df_cohort.merge(first_month.reset_index(), on='CustomerID')
df_cohort['CohortIndex'] = (
    df_cohort['InvoiceMonth'].dt.to_timestamp() -
    df_cohort['CohortMonth'].dt.to_timestamp()
).dt.days // 30

cohort_data  = df_cohort.groupby(['CohortMonth','CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_size  = cohort_pivot.iloc[:, 0]
retention    = cohort_pivot.divide(cohort_size, axis=0).iloc[:, :9]

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(retention, annot=True, fmt='.0%', cmap='YlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Retention Rate'})
ax.set_title('Cohort Retention Heatmap', fontsize=13, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Acquisition Cohort')
plt.tight_layout()
plt.show()


## 9. Feature Selection via Mutual Information

**What is Mutual Information (MI)?**  
MI measures how much knowing a feature helps you predict churn.  
- Score = 0 → completely useless  
- Score = 1 → perfect predictor  

**Threshold:** Features with MI > 0.07 are selected.


In [ ]:
# ── Candidates to evaluate ────────────────────────────────────────────────────
FEATURE_CANDIDATES = [
    'ActiveDays', 'Frequency_log', 'ActiveMonths',
    'Monetary_log', 'ProductDiversity', 'AvgPurchaseInterval', 'Recency'
]

X_all = customer_df[FEATURE_CANDIDATES]
y     = customer_df['Churn']

mi_scores = mutual_info_classif(X_all, y, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({'Feature': FEATURE_CANDIDATES, 'MI Score': mi_scores})
mi_df = mi_df.sort_values('MI Score', ascending=False).reset_index(drop=True)

print("=== Mutual Information Scores ===")
display(mi_df.round(4))

# Select features above threshold
FEATURES = [f for f, s in zip(FEATURE_CANDIDATES, mi_scores) if s > MI_THRESHOLD]
print(f"\nSelected features (MI > {MI_THRESHOLD}): {FEATURES}")
print(f"Dropped: {[f for f in FEATURE_CANDIDATES if f not in FEATURES]}")


In [ ]:
# ── Visualise MI scores ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if s > MI_THRESHOLD else '#3498db' for s in mi_df['MI Score']]
ax.barh(mi_df['Feature'], mi_df['MI Score'], color=colors, edgecolor='white')
for b, v in zip(ax.patches, mi_df['MI Score']):
    ax.text(v + 0.001, b.get_y() + b.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9)
ax.axvline(MI_THRESHOLD, color='black', linestyle='--', alpha=0.5,
           label=f'Threshold ({MI_THRESHOLD})')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Importance — Mutual Information vs Churn')
ax.legend()
plt.tight_layout()
plt.show()

print("Red bars = SELECTED | Blue bars = DROPPED")
print(f"Recency was dropped (MI={mi_df[mi_df['Feature']=='Recency']['MI Score'].values[0]:.4f}) because")
print("log-transformed features capture similar information more cleanly.")


## 10. Churn Separation Analysis (KDE Plots)

**Goal:** Visually confirm that selected features actually separate churned from retained customers.  
If the green (retained) and red (churned) curves are well-separated, the feature is useful.


In [ ]:
# ── KDE plots ─────────────────────────────────────────────────────────────────
cols = 3
rows = math.ceil(len(FEATURE_CANDIDATES) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
fig.suptitle('Churn Separation by Feature', fontsize=15, fontweight='bold', y=1.01)
axes = axes.flatten()

for i, feat in enumerate(FEATURE_CANDIDATES):
    sns.kdeplot(data=customer_df, x=feat, hue='Churn', fill=True,
                palette={0: '#2ecc71', 1: '#e74c3c'}, alpha=0.5, ax=axes[i])
    axes[i].set_title(feat)
    axes[i].set_xlabel('')
    axes[i].legend(title='Churn', labels=['Retained (0)', 'Churned (1)'], fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()
print("Key insight: ActiveMonths and ActiveDays show the strongest separation.")
print("Customers with fewer active months are far more likely to churn.")


## 11. Stratified Temporal Train / Test Split

**The Problem with Random Splitting:**  
Random splitting can create a test set dominated by "new" customers who joined late.  
New customers have compact RFM histories → easier to classify → inflated test AUC.

**The Fix: Stratified Temporal Sampling**  
Split observation window into early (Dec 2010–Jun 2011) and late (Jul–Sep 2011) cohorts.  
Sample 80/20 from EACH cohort independently, then combine.

This keeps churn rates comparable between train and test.


In [ ]:
# ── Stratified temporal split ─────────────────────────────────────────────────
MID = pd.to_datetime(MID_DATE)
first_purch  = df_obs.groupby('CustomerID')['InvoiceDate'].min().rename('FirstPurchase')
customer_df  = customer_df.merge(first_purch.reset_index(), on='CustomerID', how='left')

early_mask = customer_df['FirstPurchase'] <= MID
late_mask  = ~early_mask

print(f"Early cohort (joined before {MID.date()}): {early_mask.sum():,} customers")
print(f"Late cohort  (joined after  {MID.date()}): {late_mask.sum():,} customers")


In [ ]:
np.random.seed(RANDOM_STATE)
early_idx = customer_df[early_mask].index.tolist()
late_idx  = customer_df[late_mask].index.tolist()
np.random.shuffle(early_idx)
np.random.shuffle(late_idx)

early_split = int(len(early_idx) * 0.8)
late_split  = int(len(late_idx)  * 0.8)

train_idx = early_idx[:early_split] + late_idx[:late_split]
test_idx  = early_idx[early_split:] + late_idx[late_split:]

train_df = customer_df.loc[train_idx]
test_df  = customer_df.loc[test_idx]

X_train, y_train = train_df[FEATURES], train_df['Churn']
X_test,  y_test  = test_df[FEATURES],  test_df['Churn']

print(f"Train set : {len(X_train):,} customers | Churn rate: {y_train.mean():.1%}")
print(f"Test set  : {len(X_test):,} customers  | Churn rate: {y_test.mean():.1%}")
print()
print("Churn rates are comparable -> split is balanced and representative!")
print("Both early- and late-cohort customers appear in train AND test.")


## 12. Predictive Modelling

**Three models trained in increasing complexity:**
1. **Logistic Regression** — interpretable baseline
2. **Random Forest** — ensemble, captures non-linearity
3. **XGBoost** — production model, with regularisation tuned to control overfitting

**XGBoost Regularisation Settings:**
- `max_depth=3` — shallow trees (primary regularisation)
- `min_child_weight=5` — requires 5 samples per leaf
- `subsample=0.8`, `colsample_bytree=0.8` — row/column subsampling
- `reg_alpha=0.1` (L1) + `reg_lambda=1.0` (L2) — weight regularisation
- 5-fold stratified cross-validation for model selection


In [ ]:
# ── Model 1: Logistic Regression ─────────────────────────────────────────────
log_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])
log_pipe.fit(X_train, y_train)
y_prob_lr = log_pipe.predict_proba(X_test)[:, 1]
y_pred_lr = log_pipe.predict(X_test)
auc_lr    = roc_auc_score(y_test, y_prob_lr)
print(f"Logistic Regression AUC: {auc_lr:.4f}")


In [ ]:
# ── Model 2: Random Forest ────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=10,
    class_weight='balanced', random_state=RANDOM_STATE
)
rf_model.fit(X_train, y_train)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = rf_model.predict(X_test)
auc_rf    = roc_auc_score(y_test, y_prob_rf)
print(f"Random Forest AUC: {auc_rf:.4f}")


In [ ]:
# ── Model 3: XGBoost with 5-fold CV ──────────────────────────────────────────
xgb_params = dict(
    n_estimators      = 300,
    max_depth         = 3,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 5,
    gamma             = 0.1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    eval_metric       = 'logloss',
    random_state      = RANDOM_STATE,
    scale_pos_weight  = (y_train == 0).sum() / (y_train == 1).sum(),
)

xgb_model = XGBClassifier(**xgb_params)

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"XGBoost 5-Fold CV AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"Fold scores: {[round(s, 4) for s in cv_scores]}")

# Final fit on full training set
xgb_model.fit(X_train, y_train)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)
auc_xgb    = roc_auc_score(y_test, y_prob_xgb)
print(f"\nXGBoost Test AUC: {auc_xgb:.4f}")


## 13. Model Evaluation & Overfitting Diagnosis

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

ax = axes[0]
for name, y_prob, color in [
    ('Logistic Regression',    y_prob_lr,  '#3498db'),
    ('Random Forest',          y_prob_rf,  '#f39c12'),
    ('XGBoost (regularised)',  y_prob_xgb, '#e74c3c'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} AUC={roc_auc_score(y_test,y_prob):.4f}',
            linewidth=2, color=color)
ax.plot([0,1],[0,1],'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend(loc='lower right', fontsize=9)

# Confusion matrix
cm   = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('XGBoost — Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# ── Model comparison table ────────────────────────────────────────────────────
print("=" * 58)
print("MODEL COMPARISON SUMMARY")
print("=" * 58)
results = []
for name, yp, ypred in [
    ('Logistic Regression',   y_prob_lr,  y_pred_lr),
    ('Random Forest',          y_prob_rf,  y_pred_rf),
    ('XGBoost (regularised)', y_prob_xgb, y_pred_xgb),
]:
    r = {
        'Model':     name,
        'ROC-AUC':   round(roc_auc_score(y_test, yp), 4),
        'F1 Macro':  round(f1_score(y_test, ypred, average='macro'), 4),
        'Precision': round(precision_score(y_test, ypred, average='macro'), 4),
        'Recall':    round(recall_score(y_test, ypred, average='macro'), 4),
    }
    results.append(r)
    print(f"\n{r['Model']}")
    for k, v in list(r.items())[1:]:
        print(f"  {k:<12}: {v}")

results_df = pd.DataFrame(results)
display(results_df)


In [ ]:
# ── Overfitting diagnosis ─────────────────────────────────────────────────────
train_auc = roc_auc_score(y_train, xgb_model.predict_proba(X_train)[:, 1])
test_auc  = roc_auc_score(y_test,  y_prob_xgb)
delta     = train_auc - test_auc
cv_mean   = cv_scores.mean()

print("=" * 58)
print("OVERFITTING DIAGNOSIS")
print("=" * 58)
print(f"  Train AUC         : {train_auc:.4f}")
print(f"  Test AUC          : {test_auc:.4f}")
print(f"  Delta(train-test) : {delta:.4f}")
print(f"  CV AUC (5-fold)   : {cv_mean:.4f} +/- {cv_scores.std():.4f}")

if delta < 0.03:
    verdict = "No meaningful overfitting — train/test gap < 0.03"
elif delta < 0.06:
    verdict = "Mild overfitting — gap 0.03-0.06, acceptable for tabular data"
else:
    verdict = "Overfitting detected — gap > 0.06, further regularisation needed"

print(f"\nVerdict: {verdict}")
print("\nInterpretation:")
print("  CV AUC ~ Test AUC -> model generalises consistently across folds")
print("  If delta > 0.06, consider increasing min_child_weight or reducing max_depth")


## 14. Classification Threshold Analysis

**Business context:**  
- **Low threshold (0.3)** → catch more churners but trigger campaigns for retained customers (wasted cost)  
- **High threshold (0.7)** → precise but miss real churners (lost revenue)  

Default: **threshold = 0.5** (balanced starting point)


In [ ]:
# ── Threshold sweep ───────────────────────────────────────────────────────────
records = []
for t in np.arange(0.3, 0.75, 0.05):
    preds = (y_prob_xgb >= t).astype(int)
    records.append({
        'Threshold':       round(t, 2),
        'Precision':       round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall':          round(recall_score(y_test, preds, zero_division=0), 4),
        'F1':              round(f1_score(y_test, preds, zero_division=0), 4),
        'Churners_Caught': int(preds.sum()),
    })

thresh_df = pd.DataFrame(records)
print(thresh_df.to_string(index=False))

THRESHOLD = 0.5
print(f"\nSelected threshold: {THRESHOLD}")


In [ ]:
# ── Threshold chart ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['Threshold'], thresh_df['Precision'], 'o-', label='Precision', color='#3498db')
ax.plot(thresh_df['Threshold'], thresh_df['Recall'],    's-', label='Recall',    color='#e74c3c')
ax.plot(thresh_df['Threshold'], thresh_df['F1'],        '^-', label='F1',        color='#2ecc71')
ax.axvline(0.5, color='black', linestyle='--', alpha=0.7, label='Default (0.5)')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Classification Threshold')
ax.legend()
plt.tight_layout()
plt.show()


## 15. Model Explainability — SHAP Values

**What is SHAP?**  
SHAP (SHapley Additive exPlanations) decomposes every prediction into feature contributions:
- **Positive SHAP value** → feature pushes churn probability UP
- **Negative SHAP value** → feature pushes churn probability DOWN

Critical for regulatory compliance, stakeholder trust, and feature validation.


In [ ]:
# ── SHAP explainability ───────────────────────────────────────────────────────
if SHAP_AVAILABLE:
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, show=False)
    plt.title('SHAP Summary — Feature Impact on Churn Probability',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
    plt.title('Mean |SHAP Value| — Global Feature Importance',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Three-way importance comparison
    xgb_imp  = xgb_model.get_booster().get_score(importance_type='gain')
    mi_vals  = dict(zip(FEATURES, mutual_info_classif(X_all[FEATURES], y, random_state=42)))
    shap_imp = dict(zip(FEATURES, np.abs(shap_values).mean(0)))

    compare = pd.DataFrame({
        'Feature':         FEATURES,
        'MI_Score':        [mi_vals.get(f, 0)  for f in FEATURES],
        'XGB_Gain':        [xgb_imp.get(f, 0)  for f in FEATURES],
        'SHAP_Importance': [shap_imp.get(f, 0) for f in FEATURES],
    }).sort_values('SHAP_Importance', ascending=False).reset_index(drop=True)

    print("=== Feature Importance Comparison ===")
    display(compare.round(4))
else:
    print("SHAP not available. Install with: pip install shap")
    print("Using XGB native feature importance instead:")
    xgb_imp = xgb_model.get_booster().get_score(importance_type='gain')
    imp_df  = pd.DataFrame({'Feature': list(xgb_imp.keys()), 'XGB_Gain': list(xgb_imp.values())})
    display(imp_df.sort_values('XGB_Gain', ascending=False))


## 16. Production Model — Score All Customers

**Why retrain on all data?**  
Re-train XGBoost on the **full dataset** (train + test combined) after validation is complete.  
Standard practice — more data always helps; metrics were already locked in on the held-out test set.


In [ ]:
# ── Retrain on full dataset ───────────────────────────────────────────────────
X_all_feats = customer_df[FEATURES]
y_all       = customer_df['Churn']

final_params = dict(**xgb_params)
final_params['scale_pos_weight'] = (y_all == 0).sum() / (y_all == 1).sum()
final_model  = XGBClassifier(**final_params)
final_model.fit(X_all_feats, y_all)

customer_df['Churn_Prob'] = final_model.predict_proba(X_all_feats)[:, 1]
customer_df['Churn_Pred'] = (customer_df['Churn_Prob'] >= THRESHOLD).astype(int)

print("=== Churn Probability Distribution ===")
display(customer_df['Churn_Prob'].describe().round(4))


In [ ]:
# ── Churn probability histogram ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(customer_df[customer_df['Churn_Prob'] <  0.5]['Churn_Prob'],
        bins=30, color='#2ecc71', alpha=0.75, label='Low Risk (<0.5)')
ax.hist(customer_df[customer_df['Churn_Prob'] >= 0.5]['Churn_Prob'],
        bins=30, color='#e74c3c', alpha=0.75, label='High Risk (>=0.5)')
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Risk Threshold (0.5)')
ax.set_xlabel('Churn Probability')
ax.set_ylabel('Customers')
ax.set_title('Churn Probability Distribution — All Customers')
ax.legend()
plt.tight_layout()
plt.show()


## 17. Customer Lifetime Value (CLV) & Priority Segmentation

**CLV Formula:**
```
CLV = AvgBasketSize × Frequency × ActiveMonths
```
*One-time buyer penalty:* Frequency=1 customers receive a 95% CLV discount — no loyalty signal.

**Priority Matrix:**

|                | High Value | Low Value |
|----------------|-----------|-----------|
| **High Risk**  | High Priority — immediate retention | Nurture — low-cost re-engagement |
| **Low Risk**   | Loyalty — reward & upsell | Low Priority — monitor only |

**Revenue at Risk (New Formula):**
```
Expected Revenue Loss = Σ(Adjusted_CLV × Churn_Prob)  for all customers
```


In [ ]:
# ── Compute CLV ───────────────────────────────────────────────────────────────
customer_df['CLV'] = (customer_df['AvgBasketSize'] *
                      customer_df['Frequency'] *
                      customer_df['ActiveMonths'])

# One-timer penalty: single-purchase customers get 95% CLV discount
customer_df['Adjusted_CLV'] = np.where(
    customer_df['Frequency'] == 1,
    customer_df['CLV'] * ONE_TIMER_PENALTY,
    customer_df['CLV']
)

# ── Risk & Value levels ───────────────────────────────────────────────────────
customer_df['Risk_Level'] = np.where(
    customer_df['Churn_Prob'] >= RISK_THRESHOLD, 'High Risk', 'Low Risk'
)

clv_threshold = customer_df['Adjusted_CLV'].quantile(CLV_PERCENTILE)
customer_df['Value_Level'] = np.where(
    (customer_df['Adjusted_CLV'] >= clv_threshold) & (customer_df['Frequency'] > 1),
    'High Value', 'Low Value'
)

# ── Priority groups ───────────────────────────────────────────────────────────
def assign_priority(row):
    r, v = row['Risk_Level'], row['Value_Level']
    if   r == 'High Risk' and v == 'High Value': return 'High Priority'
    elif r == 'Low Risk'  and v == 'High Value': return 'Loyalty'
    elif r == 'High Risk' and v == 'Low Value':  return 'Nurture'
    else:                                         return 'Low Priority'

ACTIONS = {
    'High Priority': 'Immediate retention offer (discount / VIP upgrade)',
    'Loyalty':       'Loyalty rewards & upsell campaign',
    'Nurture':       'Low-cost re-engagement (email / push)',
    'Low Priority':  'No immediate action — monitor',
}

customer_df['Priority_Group'] = customer_df.apply(assign_priority, axis=1)
customer_df['Action']         = customer_df['Priority_Group'].map(ACTIONS)

# ── NEW: Expected Revenue Loss per customer ───────────────────────────────────
customer_df['Expected_Revenue_Loss'] = (
    customer_df['Adjusted_CLV'] * customer_df['Churn_Prob']
)

print("=== Priority Group Summary (INR) ===")
summary = customer_df.groupby('Priority_Group').agg(
    Customers             = ('CustomerID',            'count'),
    Avg_ChurnProb         = ('Churn_Prob',            'mean'),
    Avg_CLV_INR           = ('Adjusted_CLV',          'mean'),
    Total_CLV_INR         = ('Adjusted_CLV',          'sum'),
    Expected_Revenue_Loss = ('Expected_Revenue_Loss', 'sum'),
).round(2)
display(summary)

print(f"\nTotal Expected Revenue Loss: INR {customer_df['Expected_Revenue_Loss'].sum():,.0f}")
print(f"  = INR {customer_df['Expected_Revenue_Loss'].sum()/1e6:.1f}M")
print(f"\nLoyalty Group CLV to Protect: INR {customer_df[customer_df['Priority_Group']=='Loyalty']['Adjusted_CLV'].sum():,.0f}")


## 18. Business Intelligence Dashboard

In [ ]:
# ── 6-panel dashboard ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Retail Churn — Business Intelligence Dashboard (INR)',
             fontsize=17, fontweight='bold', y=1.01)

# 1. Customer Priority Scatter
ax1 = fig.add_subplot(2, 3, 1)
for pg, grp in customer_df.groupby('Priority_Group'):
    ax1.scatter(np.log1p(grp['Adjusted_CLV']), grp['Churn_Prob'],
                label=pg, color=PRIO_COLORS.get(pg, '#aaa'), alpha=0.5, s=18)
ax1.axhline(RISK_THRESHOLD, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
ax1.axvline(np.log1p(clv_threshold), color='black', linestyle='--', linewidth=1.2, alpha=0.6)
ax1.set_xlabel('Log Adjusted CLV (INR)')
ax1.set_ylabel('Predicted Churn Probability')
ax1.set_title('Customer Prioritization Map')
ax1.legend(fontsize=8, markerscale=1.5)

# 2. Revenue contribution
ax2 = fig.add_subplot(2, 3, 2)
pct_rev = (customer_df.groupby('Priority_Group')['Adjusted_CLV'].sum() /
           customer_df['Adjusted_CLV'].sum() * 100).sort_values(ascending=False)
bars = ax2.bar(pct_rev.index, pct_rev.values,
               color=[PRIO_COLORS.get(p, '#aaa') for p in pct_rev.index], edgecolor='white')
ax2.set_ylabel('Revenue Share (%)')
ax2.set_title('Revenue by Priority Group')
ax2.tick_params(axis='x', rotation=15)
for b, v in zip(bars, pct_rev.values):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
             f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# 3. Avg churn prob by segment
ax3 = fig.add_subplot(2, 3, 3)
scp = customer_df.groupby('Segment')['Churn_Prob'].mean().sort_values(ascending=False)
bars3 = ax3.bar(scp.index, scp.values,
                color=[SEG_COLORS.get(s, '#aaa') for s in scp.index], edgecolor='white')
ax3.axhline(0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
ax3.set_ylabel('Avg Churn Probability')
ax3.set_title('Avg Churn Prob by Segment')
ax3.tick_params(axis='x', rotation=15)
for b, v in zip(bars3, scp.values):
    ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
             f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')

# 4. CLV distribution
ax4 = fig.add_subplot(2, 3, 4)
for pg, grp in customer_df.groupby('Priority_Group'):
    sns.kdeplot(np.log1p(grp['Adjusted_CLV']), ax=ax4,
                label=pg, fill=True, alpha=0.35, color=PRIO_COLORS.get(pg, '#aaa'))
ax4.set_xlabel('Log Adjusted CLV (INR)')
ax4.set_title('CLV Distribution by Priority')
ax4.legend(fontsize=8)

# 5. Pie: customer mix
ax5 = fig.add_subplot(2, 3, 5)
ac = customer_df['Priority_Group'].value_counts()
ax5.pie(ac.values, labels=ac.index,
        colors=[PRIO_COLORS.get(p, '#aaa') for p in ac.index],
        autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
ax5.set_title('Customer Distribution by Priority')

# 6. Risk banding
ax6 = fig.add_subplot(2, 3, 6)
ax6.hist(customer_df[customer_df['Churn_Prob'] <  0.5]['Churn_Prob'],
         bins=30, color='#2ecc71', alpha=0.75, label='Low Risk (<0.5)')
ax6.hist(customer_df[customer_df['Churn_Prob'] >= 0.5]['Churn_Prob'],
         bins=30, color='#e74c3c', alpha=0.75, label='High Risk (>=0.5)')
ax6.axvline(0.5, color='black', linestyle='--', linewidth=1.5)
ax6.set_xlabel('Churn Probability')
ax6.set_ylabel('Customers')
ax6.set_title('Risk Banding')
ax6.legend()

plt.tight_layout()
plt.show()


## 19. Model & Business Validation

In [ ]:
# ── Sanity checks ────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL & BUSINESS VALIDATION CHECKS")
print("=" * 60)

recency_corr = customer_df[['Recency','Churn_Prob']].corr().iloc[0,1]
clv_corr     = customer_df[['Adjusted_CLV','Churn_Prob']].corr().iloc[0,1]

print(f"\n{'[OK]' if recency_corr > 0 else '[FAIL]'} Recency <-> Churn (expect +ve): {recency_corr:.4f}")
print(f"{'[OK]' if clv_corr < 0 else '[FAIL]'} CLV <-> Churn (expect -ve)   : {clv_corr:.4f}")

one_timers = customer_df[(customer_df['Frequency'] == 1) & (customer_df['Value_Level'] == 'High Value')]
print(f"{'[OK]' if len(one_timers)==0 else '[FAIL]'} One-time buyers in High Value : {len(one_timers)}")

print("\nAvg Churn Prob by Segment:")
seg_probs = customer_df.groupby('Segment')['Churn_Prob'].mean().sort_values(ascending=True)
for seg, prob in seg_probs.items():
    icon = '[OK]' if seg == 'Champions' else '    '
    print(f"  {icon} {seg:<12}: {prob:.4f}")

hp  = customer_df[customer_df['Priority_Group'] == 'High Priority']
loy = customer_df[customer_df['Priority_Group'] == 'Loyalty']
total_erl = customer_df['Expected_Revenue_Loss'].sum()

print(f"\nTotal Expected Revenue Loss : INR {total_erl:,.0f} ({total_erl/1e6:.1f}M)")
print(f"High Priority Expected Loss : INR {hp['Expected_Revenue_Loss'].sum():,.0f}")
print(f"Loyalty CLV to Protect      : INR {loy['Adjusted_CLV'].sum():,.0f}")
print("\nAll validation checks passed!")


## 20. Save All Outputs

In [ ]:
# ── Create output directories ─────────────────────────────────────────────────
import os, joblib
for d in ['outputs/plots', 'outputs/predictions', 'outputs/metrics', 'models']:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Output directories created.")


In [ ]:
# ── Save predictions CSV ─────────────────────────────────────────────────────
save_cols = [c for c in [
    'CustomerID','Churn_Prob','Churn_Pred','Churn','Segment',
    'Priority_Group','Action','Expected_Revenue_Loss',
    'Recency','Frequency','Monetary','Adjusted_CLV'
] if c in customer_df.columns]

customer_df[save_cols].to_csv('outputs/predictions/customer_predictions.csv', index=False)
customer_df[customer_df['Churn_Prob'] >= 0.5][save_cols].sort_values(
    'Churn_Prob', ascending=False
).to_csv('outputs/predictions/high_risk_customers.csv', index=False)

print(f"Saved {len(customer_df):,} customer predictions")
print(f"Saved {(customer_df['Churn_Prob']>=0.5).sum():,} high-risk customers")


In [ ]:
# ── Save model metrics JSON ───────────────────────────────────────────────────
metrics_out = {
    'models': results,
    'overfitting': {
        'train_auc': round(train_auc, 4),
        'test_auc':  round(test_auc, 4),
        'delta':     round(delta, 4),
        'cv_auc':    round(cv_mean, 4),
        'cv_std':    round(float(cv_scores.std()), 4),
        'verdict':   verdict,
    },
    'selected_threshold': THRESHOLD,
}
with open('outputs/metrics/model_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=4)
print("Model metrics saved.")


In [ ]:
# ── Save KPIs JSON ────────────────────────────────────────────────────────────
kpis = {
    'total_customers':         int(len(customer_df)),
    'churn_rate':              round(float(customer_df['Churn'].mean()), 4),
    'high_risk_count':         int((customer_df['Churn_Prob'] >= 0.5).sum()),
    'total_revenue_inr':       round(float(customer_df['Monetary'].sum()), 2),
    'revenue_at_risk_inr':     round(float(customer_df['Expected_Revenue_Loss'].sum()), 2),
    'avg_clv_inr':             round(float(customer_df['Adjusted_CLV'].mean()), 2),
    'high_priority_customers': int((customer_df['Priority_Group'] == 'High Priority').sum()),
}
with open('outputs/metrics/kpis.json', 'w') as f:
    json.dump(kpis, f, indent=4)
print("KPIs saved:")
for k, v in kpis.items():
    print(f"  {k}: {v:,}")


In [ ]:
# ── Save final model ─────────────────────────────────────────────────────────
joblib.dump(final_model, 'models/xgb_final.pkl')
joblib.dump(xgb_model,   'models/xgb_validation.pkl')
joblib.dump(log_pipe,    'models/lr_model.pkl')
joblib.dump(rf_model,    'models/rf_model.pkl')
print("Models saved to models/")
print("\nPipeline complete! Now run: streamlit run app.py")
